# 05 · Error analysis: why is it wrong?

Notebook 04 gave a score. A score says *how often* the model is wrong, not *why*, and the why decides what
the next version should change. So between versions this project looked hard at the errors and tested each
specific worry with a small script. This notebook runs three of them:

| section | script | the question it answers | runs |
|---|---|---|---|
| 1 | `make_demo_figure.py` | What do right and wrong predictions look like? | always |
| 2 | `probe_memorization.py` | Has it memorized its training scenes, or its training signs? | stored result; re-probe with `RUN_FOR_REAL` |
| 3 | `shortcut_check.py` | Does it fire on pasted things that aren't signs? | always |

Section 2 is how v1.5b came about: the probe found scene memorization, and v1.5b's fresh scenes (notebook 01,
section 6) removed it. Section 3's question came back in v3a and led to v3a.1's decoys (notebook 07).

Two more checks of this kind live where their question comes up: blanking the sign on real photos
(notebook 06) and labeling real false alarms by hand (P6, notebook 07).

**Needs:** `data/processed/val.npz`, a trained run, and the raw GTSRB and COCO downloads (section 3 builds new
scenes). **Writes:** figures into `outputs/notebooks/figures/`; a probe result only if you turn on
`RUN_FOR_REAL`.

In [ ]:
# ---- Settings: the only cell you need to edit ----
RUN_NAME = "peak_fresh_aug"  # the run to analyze (v1.5b seed 0, as in notebook 04); try "baseline" (v1) or "peak_decoy" (v3a.1)
RUN_FOR_REAL = False  # True: re-probe RUN_NAME for memorization, writing outputs/eval/<run>/best_probe.json (~1 min)

In [ ]:
# Setup (the same in every notebook). All paths in this project (configs/, data/, outputs/) are relative
# to the repo root, and the scripts expect to run from there, so move this notebook's working folder up
# to the root first. Safe to run twice.
import os
import sys
from pathlib import Path

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists():  # walk up until we reach the folder holding pyproject.toml
    if root == root.parent:
        raise RuntimeError("can't find the repo root: open this notebook from inside FindTheObject-CNN-Project")
    root = root.parent
os.chdir(root)
if str(root / "notebooks") not in sys.path:
    sys.path.insert(0, str(root / "notebooks"))  # lets us import notebooks/nbhelpers.py

from nbhelpers import NB_OUT, ROOT, find_run, read_json, require, run_script
print("working in", ROOT)

In [ ]:
# Find the run, and set up a folder for figures plus a helper that shows a PNG file in the notebook.
from IPython.display import Image, display

require("data/processed/val.npz", how="python scripts/make_dataset.py --config configs/baseline.yaml")
RUN_DIR = find_run(RUN_NAME)
CKPT = (RUN_DIR / "best.pt").relative_to(ROOT)  # relative, as the scripts record it in their JSON
FIG = (NB_OUT / "figures").relative_to(ROOT)  # relative, so the printed commands stay short


def show(png, width=900):
    """Display a PNG (path relative to the repo root) at the given width in pixels."""
    display(Image(filename=str(ROOT / png), width=width))


print("checkpoint:", CKPT)
print("figures go to:", FIG)

## 1. Right and wrong, side by side (`make_demo_figure.py`)

Scores all of val, then draws 8 random correct scenes and 4 random wrong ones. It's the figure in the
project README, made here for whichever run you pick. The `--out` is given on purpose: the script's default
would overwrite the README's `docs/demo_val.png`.

Look at the wrong ones and ask what they have in common. For v1 the answer was size and darkness: signs of
16–23 px were 23% of val scenes but 53% of its errors, and near-black signs 22% of scenes but 42% of errors.
Those are what real road photos look like, so they were kept, not filtered out.

In [ ]:
# Draw 8 right + 4 wrong val predictions for RUN_NAME (~10 s).
run_script("make_demo_figure", "--checkpoint", CKPT, "--n-right", 8, "--n-wrong", 4, "--seed", 0,
           "--out", f"{FIG}/demo_val.png")
show(f"{FIG}/demo_val.png", width=700)

## 2. Memorization (`probe_memorization.py`)

v1.5 reached 100% class accuracy on its training scenes by epoch 9, while val stayed at 97.3%. A gap like that
means the model knows something about its training data that doesn't carry over to new data. It could be
either of two things:

* **scene memorization**: it remembers the exact training images.
* **sign memorization**: it knows the particular physical signs it trained on, not signs in general. GTSRB
  has several photos of each physical sign (a *track*), and val uses different physical signs from training.

The probe tells them apart. It scores 5,000 *new* scenes built from **training** signs and backgrounds, which
sit between the two:

| v1.5 on | joint | gap to the row above |
|---|---|---|
| its own training scenes | 100% | |
| new scenes, training signs | 98.40% | 1.60 pts: scene memorization |
| val (new scenes, new signs) | 97.30% | 1.10 pts: sign memorization |

The first gap is why v1.5b makes fresh scenes every epoch: if no scene ever repeats, there is nothing to
memorize scene by scene. The second gap, the **sign gap**, is what's left to watch. Classes with few training
tracks (≤ 12) are reported separately as **weak classes**, because that's where memorizing is easiest.

In [ ]:
# Stored probe results for every run that has one: joint accuracy on new scenes with training signs vs val.
EVAL = ROOT / "outputs/eval"
print(f"{'run':34} {'train signs':>11} {'val':>7} {'sign gap':>9}  {'weak classes: train signs':>25} {'val':>6}")
for path in sorted(EVAL.glob("*/best_probe.json")):
    p = read_json(path)
    print(f"{p['run_id']:34} {p['fresh_train_signs']['joint_acc']:11.4f} {p['val']['joint_acc']:7.4f} "
          f"{p['sign_gap_joint']:9.4f}  {p['weak_class_acc']['fresh']:25.4f} {p['weak_class_acc']['val']:6.4f}")

`peak` is v1.5, the probe above. The `peak_fresh` rows (v1.5b's run A, fresh scenes without augmentation) and
`peak_fresh_aug` rows (run B, with augmentation: the v1.5b that was kept, and its two extra seeds) come
after the fix. The scene gap is gone: `RESULTS.md` records run A at 99.5% on its own training scenes in its
last epoch and 99.6% on new scenes with training signs. The sign gap isn't: it's 0.6–1.3 points across these
runs, against v1.5's 1.10. On the weak classes, v1.5b seed 0 scores 99.5% with training signs vs 97.0% on
val, a 2.5-point gap. Some sign memorization remains, concentrated where there are few signs to learn from.

In [ ]:
# Probe RUN_NAME (writes outputs/eval/<run>/best_probe.json; ~1 min). Only runs when RUN_FOR_REAL = True.
run_script("probe_memorization", "--checkpoint", CKPT, "--n", 5000, "--seed", 987654, enabled=RUN_FOR_REAL)

## 3. Does it fire on non-signs? (`shortcut_check.py`)

Every training scene is a sign *pasted* onto a photo, so the pasted patch has a sharp edge (a **seam**) where
the photo around it changes. A model could score well by finding the seam instead of the sign: "pasted
rectangle = sign". This check, first run on v1, tests that directly. It builds 1,000 new backgrounds and
renders each in several versions that differ only in what's pasted on: nothing, a real sign, or a non-sign
rectangle pasted exactly the way signs are:

* `bg_patch`: a crop of another photo, the closest thing to a sign-shaped seam with no sign in it,
* `shuffled_sign`: a sign's pixels scrambled, so the same colors with no shape,
* `flat`: a solid patch in a sign's average color,
* `other_sign`: a second real sign, which *should* fire (a positive control).

"Fires" here means the heatmap peak is above **0.5**, a fixed cutoff. (The tuned cutoff τ arrives in
notebook 07.)

In [ ]:
# Run the shortcut check on RUN_NAME with 1,000 backgrounds (~20 s); it prints a table and draws a panel.
run_script("shortcut_check", "--checkpoint", CKPT, "--n", 1000, "--seed", 12345,
           "--panel-out", f"{FIG}/shortcut_check.png")
show(f"{FIG}/shortcut_check.png")

**Reading the panel:** each row is one background. Columns come in pairs, the scene and then its heatmap in
red: *sign + decoy*, *decoy only*, *nothing pasted*. Green box = real sign, red box = decoy. A good model glows
inside the green box only.

**Reading the table:** `peak>0.5` is the share of scenes where the model fires.

* **v1** (`RUN_NAME = "baseline"`) passed: `bg_patch` fired in 2.1% of scenes, against 98.9% for a real sign.
  The one leak was solid patches: `flat` fired in 26% of scenes, and 59% of the time when the patch was dark,
  because 21% of training signs are nearly black.
* **v1.5b** (the default) fires more: about 12% on `bg_patch`, 15% on `shuffled_sign` and 27% on `flat`. The
  project never pinned down which change between v1 and v1.5b caused that.

At the time it didn't matter much: v1.5b never has to say "no sign", because every scene it sees has one.
It started to matter in v3a, which adds empty scenes. v3a tested the same worry on its own decoy scenes
(prediction P7) and failed it, with 9% and 12% false alarms against a 3% limit. v3a.1 then trained on decoys,
and notebook 07 tells that part. Set `RUN_NAME = "peak_decoy"` here to see the result: 1.5–2.6% on the decoy
rows, close to the 1.0% it fires on empty scenes.

**Next:** `06_real_photos.ipynb`: does a model trained on pasted signs work on real road photos?